# FineTuning FaceNet part 1

## Installing Dependencies

In [23]:
!pip uninstall -f pillow
!pip install pillow --upgrade
!pip install facenet-pytorch==2.5.3 --upgrade


Usage:   
  pip3 uninstall [options] <package> ...
  pip3 uninstall [options] -r <requirements file> ...

no such option: -f


## Imports

In [24]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from scipy.stats import mode
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from facenet_pytorch import InceptionResnetV1

## Configurations

In [35]:
class Config:
    BASE_DIR = "/kaggle/input/surveillance-for-retail-stores/face_identification/face_identification"
    TRAIN_CSV = os.path.join(BASE_DIR, "trainset.csv")
    CHECKPOINT_DIR = "/kaggle/working/checkpoints/facenet_triplet"
    BATCH_SIZE = 512  
    NUM_EPOCHS = 45
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EMBEDDING_DIM = 1024
    IMAGE_SIZE = (160, 160)
    NUM_WORKERS = 2
    SCALING_FACTOR = 500.0
    NUM_CLASSES = None

os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)
print(f"Using device: {Config.DEVICE}")

## Dataset Helper Class

In [36]:
class FaceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        try:
            img_path = self.df.iloc[idx]['full_path']
            label = self.df.iloc[idx]['label']
            image = Image.open(img_path).convert('RGB')
            image = np.array(image)
            if self.transform:
                augmented = self.transform(image=image)
                image = augmented['image']
            return image, label
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            return self.__getitem__((idx + 1) % len(self.df))

## Model Definition + Triplet Loss

In [37]:

class FaceNetModel(nn.Module):
    def __init__(self, embedding_dim=1024):
        super(FaceNetModel, self).__init__()
        self.backbone = InceptionResnetV1(pretrained='vggface2')
        self.backbone.classify = False
        self.embedding_layer = nn.Sequential(
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, embedding_dim),
            nn.BatchNorm1d(embedding_dim)
        )
    
    def forward(self, x, return_embeddings=True):
        features = self.backbone(x)
        embeddings = self.embedding_layer(features)
        scaled_embeddings = embeddings * Config.SCALING_FACTOR
        return scaled_embeddings  # No normalization here

class TripletLoss(nn.Module):
    def __init__(self, margin):
        super(TripletLoss, self).__init__()
        self.margin = margin
    
    def forward(self, anchor, positive, negative):
        anchor = F.normalize(anchor, p=2, dim=1)
        positive = F.normalize(positive, p=2, dim=1)
        negative = F.normalize(negative, p=2, dim=1)
        pos_dist = F.pairwise_distance(anchor, positive)
        neg_dist = F.pairwise_distance(anchor, negative)
        loss = F.relu(pos_dist - neg_dist + self.margin)
        return loss.mean(), pos_dist.mean(), neg_dist.mean()

def get_margin(epoch):
    if epoch < 4:
        return 1.0
    elif epoch < 7:
        return 2.0
    else:
        return 3.0

def generate_triplets(embeddings, labels, margin, epoch):
    labels = labels.detach()
    anchor, positive, negative = [], [], []
    batch_size = embeddings.size(0)
    
    dist_matrix = torch.cdist(embeddings, embeddings)
    
    hard_ratio = 0.75  # Fixed at 75%
    
    for i in range(batch_size):
        label = labels[i]
        pos_mask = (labels == label) & (torch.arange(batch_size).to(Config.DEVICE) != i)
        neg_mask = (labels != label)
        
        pos_indices = torch.where(pos_mask)[0]
        neg_indices = torch.where(neg_mask)[0]
        
        if len(pos_indices) == 0 or len(neg_indices) == 0:
            continue
        
        pos_dists = dist_matrix[i, pos_indices]
        neg_dists = dist_matrix[i, neg_indices]
        
        pos_idx = pos_indices[torch.argmax(pos_dists)]  # Hardest positive
        pos_dist = pos_dists.max()
        
        if torch.rand(1).item() < hard_ratio:
            neg_idx = neg_indices[torch.argmin(neg_dists)]  # Hardest negative
        else:
            semi_hard_mask = (neg_dists > pos_dist) & (neg_dists < pos_dist + margin)
            semi_hard_indices = neg_indices[semi_hard_mask]
            if len(semi_hard_indices) > 0:
                neg_idx = semi_hard_indices[torch.randint(0, len(semi_hard_indices), (1,)).item()]
            else:
                neg_idx = neg_indices[torch.argmin(neg_dists)]
        
        anchor.append(embeddings[i])
        positive.append(embeddings[pos_idx])
        negative.append(embeddings[neg_idx])
    
    if len(anchor) == 0:
        return None, None, None
    
    return (torch.stack(anchor), torch.stack(positive), torch.stack(negative))

def compute_class_weights(df):
    class_counts = df['label'].value_counts().sort_index()
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[df['label']].values
    print(f"Class Weights - Min: {class_weights.min():.4f}, Max: {class_weights.max():.4f}")
    return sample_weights

## Validation with clustering accuracy

In [38]:
def evaluate_embeddings(model, val_loader):
    model.eval()
    all_embeddings = []
    all_labels = []
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Generating validation embeddings"):
            images = images.to(Config.DEVICE)
            embeddings = model(images)
            all_embeddings.append(embeddings.cpu().numpy())
            all_labels.append(labels.numpy())
    all_embeddings = np.concatenate(all_embeddings)
    all_labels = np.concatenate(all_labels)
    
    embeddings_norm = all_embeddings / np.linalg.norm(all_embeddings, axis=1, keepdims=True)
    dist_matrix = torch.cdist(torch.tensor(embeddings_norm), torch.tensor(embeddings_norm)).numpy()
    pos_mask = (all_labels[:, None] == all_labels[None, :]) & ~np.eye(len(all_labels), dtype=bool)
    neg_mask = all_labels[:, None] != all_labels[None, :]
    pos_dists = dist_matrix[pos_mask]
    neg_dists = dist_matrix[neg_mask]
    
    mean_pos_dist = np.mean(pos_dists)
    mean_neg_dist = np.mean(neg_dists)
    max_pos_dist = np.max(pos_dists)
    
    kmeans = KMeans(n_clusters=Config.NUM_CLASSES, random_state=42).fit(embeddings_norm)
    pred_labels = kmeans.labels_
    label_map = {}
    for cluster in range(Config.NUM_CLASSES):
        cluster_labels = all_labels[pred_labels == cluster]
        if len(cluster_labels) > 0:  # Only process non-empty clusters
            mode_result = mode(cluster_labels)
            label_map[cluster] = mode_result.mode.item() 
    mapped_labels = [label_map.get(pred, -1) for pred in pred_labels]
    accuracy = np.mean([1 if pred == true else 0 for pred, true in zip(mapped_labels, all_labels)]) * 100
    
    print(f"Validation - Mean Pos Dist: {mean_pos_dist:.4f}, Mean Neg Dist: {mean_neg_dist:.4f}, "
          f"Max Pos Dist: {max_pos_dist:.4f}, Clustering Accuracy: {accuracy:.2f}%, "
          f"Norm Embedding Std: {np.std(embeddings_norm):.4f}, Raw Embedding Std: {np.std(all_embeddings):.4f}")
    return mean_pos_dist, mean_neg_dist, max_pos_dist, accuracy

## Training

In [39]:
# Training step
def train_step(model, images, labels, epoch, batch_idx, optimizer, criterion_triplet, train_loader):
    optimizer.zero_grad()
    embeddings = model(images)
    anchor, positive, negative = generate_triplets(embeddings, labels, criterion_triplet.margin, epoch)
    if anchor is None:
        return 0.0, 0.0, 0.0
    loss, mean_pos_dist, mean_neg_dist = criterion_triplet(anchor, positive, negative)
    if batch_idx % 10 == 0 or batch_idx == len(train_loader) - 1:
        print(f"Epoch {epoch+1}, Batch {batch_idx} - Triplet Loss: {loss.item():.4f}, "
              f"Mean Pos Dist: {mean_pos_dist.item():.4f}, Mean Neg Dist: {mean_neg_dist.item():.4f}, "
              f"Embeddings Mean: {embeddings.mean().item():.4f}, Std: {embeddings.std().item():.4f}")
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
    optimizer.step()
    return loss.item(), mean_pos_dist.item(), mean_neg_dist.item()

# Training loop
def train_model(model, train_loader, val_loader, criterion_triplet, optimizer, scheduler, num_epochs):
    best_acc = 0.0
    best_epoch = 0
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        total_pos_dist = 0.0
        total_neg_dist = 0.0
        criterion_triplet.margin = get_margin(epoch + 1)
        print(f"Epoch {epoch+1} - Margin: {criterion_triplet.margin}")
        
        for batch_idx, (images, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Training")):
            images = images.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)
            loss, pos_dist, neg_dist = train_step(model, images, labels, epoch, batch_idx, optimizer, criterion_triplet, train_loader)
            train_loss += loss
            total_pos_dist += pos_dist
            total_neg_dist += neg_dist
        
        train_loss /= len(train_loader)
        avg_pos_dist = total_pos_dist / len(train_loader)
        avg_neg_dist = total_neg_dist / len(train_loader)
        print(f"Epoch {epoch+1} Summary - Train Loss: {train_loss:.4f}, "
              f"Avg Pos Dist: {avg_pos_dist:.4f}, Avg Neg Dist: {avg_neg_dist:.4f}")
        
        mean_pos_dist, mean_neg_dist, max_pos_dist, val_acc = evaluate_embeddings(model, val_loader)
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_epoch = epoch + 1
            torch.save({'model_state_dict': model.state_dict()}, 
                       f"{Config.CHECKPOINT_DIR}/best_model_epoch_{best_epoch}.pth")
            print(f"Saved best model at epoch {best_epoch} with Clustering Accuracy: {val_acc:.2f}%")
        
        scheduler.step()
    
    return best_acc, best_epoch

# Main function
def main():
    torch.manual_seed(42)
    np.random.seed(42)
    os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)

    print("Loading training data...")
    train_df = pd.read_csv(Config.TRAIN_CSV, sep=',', engine='python', header=0, 
                          names=['image_path', 'gt'], on_bad_lines='skip')
    train_df = train_df.dropna()
    train_df['image_path'] = train_df['image_path'].str.strip()
    train_df['gt'] = train_df['gt'].str.strip()
    train_df['full_path'] = train_df['image_path'].apply(lambda x: os.path.join(Config.BASE_DIR, x))
    le = LabelEncoder()
    train_df['label'] = le.fit_transform(train_df['gt'])
    Config.NUM_CLASSES = len(le.classes_)
    print(f"Training data loaded: {len(train_df)} samples, {Config.NUM_CLASSES} classes")

    train_data, val_data = train_test_split(train_df, test_size=0.2, stratify=train_df['label'], random_state=42)
    print(f"Train split: {len(train_data)} samples, Val split: {len(val_data)} samples")

    train_transform = A.Compose([
        A.Resize(*Config.IMAGE_SIZE),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2()
    ])
    val_transform = A.Compose([
        A.Resize(*Config.IMAGE_SIZE),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2()
    ])

    train_dataset = FaceDataset(train_data, transform=train_transform)
    val_dataset = FaceDataset(val_data, transform=val_transform)

    sample_weights = compute_class_weights(train_data)
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, sampler=sampler, 
                             num_workers=Config.NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, 
                           num_workers=Config.NUM_WORKERS, pin_memory=True)

    model = FaceNetModel(embedding_dim=Config.EMBEDDING_DIM).to(Config.DEVICE)
    criterion_triplet = TripletLoss(margin=1.0)
    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[10, 20, 30], gamma=0.5)

    best_acc, best_epoch = train_model(model, train_loader, val_loader, criterion_triplet, optimizer, scheduler, Config.NUM_EPOCHS)

    print("Final embedding evaluation:")
    checkpoint = torch.load(f"{Config.CHECKPOINT_DIR}/best_model_epoch_{best_epoch}.pth")
    model.load_state_dict(checkpoint['model_state_dict'])
    mean_pos_dist, mean_neg_dist, max_pos_dist, final_acc = evaluate_embeddings(model, val_loader)
    print(f"Final Results - Mean Pos Dist: {mean_pos_dist:.4f}, Mean Neg Dist: {mean_neg_dist:.4f}, "
          f"Max Pos Dist: {max_pos_dist:.4f}, Clustering Accuracy: {final_acc:.2f}%")
    
    return model, best_acc, le, train_df

In [ ]:
if __name__ == "__main__":
    model, best_acc, le, train_df = main()